# 04 — Feature Engineering & Modelling Dataset

**CrimeMapAI | Primary dataset: 2017–2022 district-year panel**

This stage converts the validated district-year panel into a leakage-controlled modelling table. The project predicts **next-year reported cyber-crime counts**. Population-normalised rates are not constructed because no defensible population denominator is present in the current dataset.

### EDA decisions carried forward
- The target is highly zero-inflated and strongly right-skewed.
- Crime is highly concentrated: a small number of districts contribute a large share of reported offences.
- Therefore we retain the original count target for interpretable evaluation and additionally create `log1p(target)` as a modelling target option.
- Temporal features must respect actual calendar-year continuity; missing years must not be bridged.
- `id`, registration-circle identifiers, and contemporaneous/future target information are excluded from predictors.


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT = Path("../data/processed/district_year_panel.csv")
OUTPUT = Path("../data/processed/model_table_v2.csv")

assert INPUT.exists(), f"Run 02_data_preprocessing.ipynb first: {INPUT}"

df = pd.read_csv(INPUT, low_memory=False)
print("Input shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
print("Columns:", len(df.columns))


Input shape: (4347, 46)
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Columns: 46


## 1. Validate district-year grain

In [11]:
key = ["state_code", "district_code", "year"]
dups = df.duplicated(key, keep=False)
print("Duplicate district-year rows:", int(dups.sum()))
assert not dups.any(), "district_year_panel must have one row per district-year"
print("Unique district-years:", df[key].drop_duplicates().shape[0])

Duplicate district-year rows: 0
Unique district-years: 4347


## 2. Define modelling target and calendar-safe future target

In [12]:
df = df.sort_values(["state_code", "district_code", "year"]).reset_index(drop=True)

grp = df.groupby(["state_code", "district_code"], sort=False)

df["target_next_year"] = grp["total_offences_under_ip"].shift(-1)
df["next_year"] = grp["year"].shift(-1)

# Only retain a target when the next observation is exactly one calendar year later.
df.loc[df["next_year"] != df["year"] + 1, "target_next_year"] = np.nan
df.loc[df["next_year"] != df["year"] + 1, "next_year"] = np.nan

df["target_log1p"] = np.log1p(df["target_next_year"])

print("Target rows:", df["target_next_year"].notna().sum())
print("Target years:", sorted(df.loc[df["target_next_year"].notna(), "year"].unique()))
print("Missing-target rows:", df["target_next_year"].isna().sum())

Target rows: 3594
Target years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Missing-target rows: 753


## 3. Calendar-safe lag and rolling features

In [13]:
crime_cols = [
    c for c in df.columns
    if c not in {"id", "year", "state_code", "state_name", "district_code", "district_name",
                 "registration_circles"}
    and pd.api.types.is_numeric_dtype(df[c])
]

# Calendar-safe lags for ALL numeric crime variables, including the source target.
# This explicitly creates total_offences_under_ip_lag1/lag2.
base = df[["state_code", "district_code", "year"] + crime_cols].copy()
for lag in (1, 2):
    lagged = base.copy()
    lagged["year"] = lagged["year"] + lag
    rename = {c: f"{c}_lag{lag}" for c in crime_cols}
    lagged = lagged.rename(columns=rename)
    lagged = lagged[["state_code", "district_code", "year"] + list(rename.values())]
    df = df.merge(
        lagged,
        on=["state_code", "district_code", "year"],
        how="left",
        validate="one_to_one",
    )

for col in crime_cols:
    l1, l2 = f"{col}_lag1", f"{col}_lag2"
    df[f"{col}_rolling_mean2"] = df[[l1, l2]].mean(axis=1, skipna=True)
    df[f"{col}_rolling_std2"] = df[[l1, l2]].std(axis=1, skipna=False, ddof=1)

# Human-readable target-history aliases.
df["total_offences_lag1"] = df["total_offences_under_ip_lag1"]
df["total_offences_lag2"] = df["total_offences_under_ip_lag2"]
df["total_offences_yoy_change"] = (
    df["total_offences_lag1"] - df["total_offences_lag2"]
)
df["total_offences_yoy_growth"] = np.where(
    df["total_offences_lag2"] > 0,
    df["total_offences_yoy_change"] / df["total_offences_lag2"],
    np.nan,
)

print("Calendar-safe lag coverage:")
print(
    df[["year", "total_offences_lag1", "total_offences_lag2", "total_offences_yoy_growth"]]
    .notna()
    .mean()
)


Calendar-safe lag coverage:
year                         1.000000
total_offences_lag1          0.826777
total_offences_lag2          0.658615
total_offences_yoy_growth    0.301817
dtype: float64


C:\Users\nishantwhat\AppData\Local\Temp\ipykernel_31848\3791630977.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_rolling_std2"] = df[[l1, l2]].std(axis=1, skipna=False, ddof=1)
C:\Users\nishantwhat\AppData\Local\Temp\ipykernel_31848\3791630977.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_rolling_mean2"] = df[[l1, l2]].mean(axis=1, skipna=True)
C:\Users\nishantwhat\AppData\Local\Temp\ipykernel_31848\3791630977.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu

## 4. Composition and concentration features

In [14]:
# Category-composition features use only individual crime indicators,
# not the source-defined total target.
count_cols = [c for c in crime_cols if c != "total_offences_under_ip"]
row_sum = df[count_cols].sum(axis=1)

df["crime_category_total"] = row_sum
df["nonzero_category_count"] = (df[count_cols] > 0).sum(axis=1)
df["crime_category_diversity"] = df["nonzero_category_count"] / max(len(count_cols), 1)

df["total_offences_log1p"] = np.log1p(df["total_offences_under_ip"])
df["total_offences_lag1_log1p"] = np.log1p(df["total_offences_lag1"])
df["total_offences_lag2_log1p"] = np.log1p(df["total_offences_lag2"])

print(
    df[
        ["crime_category_total", "nonzero_category_count",
         "crime_category_diversity", "total_offences_log1p"]
    ].describe()
)


       crime_category_total  nonzero_category_count  crime_category_diversity  \
count           4347.000000             4347.000000               4347.000000   
mean            1754.974417                5.616747                  0.130622   
std              887.663994                4.413913                  0.102649   
min                0.000000                0.000000                  0.000000   
25%             2020.000000                2.000000                  0.046512   
50%             2032.693147                5.000000                  0.116279   
75%             2066.000000                8.000000                  0.186047   
max            12594.000000               30.000000                  0.697674   

       total_offences_log1p  
count           4347.000000  
mean               1.330759  
std                1.647407  
min                0.000000  
25%                0.000000  
50%                0.000000  
75%                2.639057  
max                8.593413  


C:\Users\nishantwhat\AppData\Local\Temp\ipykernel_31848\1355781600.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["crime_category_total"] = row_sum
C:\Users\nishantwhat\AppData\Local\Temp\ipykernel_31848\1355781600.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["nonzero_category_count"] = (df[count_cols] > 0).sum(axis=1)
C:\Users\nishantwhat\AppData\Local\Temp\ipykernel_31848\1355781600.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

## 5. Feature eligibility and leakage audit

In [15]:
DROP = {
    "id", "registration_circles", "state_name", "district_name",
    "target_next_year", "target_log1p", "next_year"
}

# Keep geography as categorical identifiers for tree models / one-hot pipelines later,
# but never use registration-circle row identifiers as predictors.
feature_cols = [c for c in df.columns if c not in DROP]

# Target itself and any future-year columns must never be features.
LEAKAGE = {"target_next_year", "target_log1p", "next_year"}
assert not LEAKAGE.intersection(feature_cols)

model_df = df[df["target_next_year"].notna()].copy()
print("Final modelling rows:", len(model_df))
print("Prediction-origin years:", sorted(model_df["year"].unique()))
print("Feature count:", len(feature_cols))

missing = model_df[feature_cols].isna().mean().sort_values(ascending=False).head(20)
print("Top feature missingness:")
print(missing)

Final modelling rows: 3594
Prediction-origin years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Feature count: 230
Top feature missingness:
total_offences_yoy_growth                                  0.738175
cheating_by_personation_by_using_computer_resource_lag2    0.406511
tampering_computer_source_documents_lag2                   0.406511
ransom_ware_lag2                                           0.406511
presrv_and_retention_of_info_by_intermediaries_lag2        0.406511
identity_theft_lag2                                        0.406511
offences_other_than_ransom_ware_lag2                       0.406511
dishonestly_recv_stolen_cmp_resrc_or_comm_device_lag2      0.406511
violation_of_privacy_lag2                                  0.406511
cyber_terrorism_lag2                                       0.406511
pub_or_trans_mtrl_cont_sxly_explct_act_elect_frm_lag2      0.406511
pub_or_trans_obscene_material_in_electronic_form_lag2      0.406511
interce

## 6. Export modelling table

In [16]:
model_df.to_csv(OUTPUT, index=False)
print("Saved:", OUTPUT)
print("Shape:", model_df.shape)
print("Target summary:")
print(model_df["target_next_year"].describe())
print("Target zero share:", (model_df["target_next_year"] == 0).mean())

Saved: ..\data\processed\model_table_v2.csv
Shape: (3594, 235)
Target summary:
count    3594.000000
mean       28.225376
std       198.460502
min         0.000000
25%         0.000000
50%         1.000000
75%        14.000000
max      5395.000000
Name: target_next_year, dtype: float64
Target zero share: 0.4986087924318308


## 7. Modelling recommendation

In [17]:
print("Recommended modelling setup:")
print("1. Baseline: persistence, target_next_year = current-year total_offences_under_ip")
print("2. Regression target: target_next_year, with target_log1p as a secondary formulation")
print("3. Validation: temporal / rolling-origin; never random split")
print("4. Headline metrics: MAE, RMSE, R²; compare raw-target vs log-target formulation")
print("5. Classification layer: derive Low/Medium/High risk from training-only target quantiles after regression pipeline is validated")

Recommended modelling setup:
1. Baseline: persistence, target_next_year = current-year total_offences_under_ip
2. Regression target: target_next_year, with target_log1p as a secondary formulation
3. Validation: temporal / rolling-origin; never random split
4. Headline metrics: MAE, RMSE, R²; compare raw-target vs log-target formulation
5. Classification layer: derive Low/Medium/High risk from training-only target quantiles after regression pipeline is validated
